# dim_clientes
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Gold (nivel 2) · dimensión Clientes (SCD2) |
| Propósito | Una fila por versión de cliente, con su segmento y vigencia (vigente_desde, vigente_hasta, es_vigente). |
| Fuentes | silver.clientes_historial (segmento versionado) + silver.clientes (datos actuales) |
| Decisiones | - La primera versión vale desde 1900: una venta anterior al primer cambio registrado también encuentra su versión. - es_posible_duplicado: misma persona con otra cuenta (mismo email canónico, o mismo nombre + apellido + teléfono). Se marca; fusionar exige una regla de identidad del negocio. |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import Window
from pyspark.sql import functions as F

from comun_gold import FIN_DE_LOS_TIEMPOS, INICIO_DE_LOS_TIEMPOS, leer_silver, sk

## 2. Tabla `dim_clientes`
Clientes con historial de segmento (SCD2): una fila por versión, con su vigencia.

In [ ]:
@dlt.table(name="dim_clientes", comment="Clientes con historial de segmento (SCD2): una fila por versión, con su vigencia.")
def dim_clientes():
    historial = leer_silver("clientes_historial")
    actual = leer_silver("clientes").select(
        "id_cliente", "nombre", "apellido", "email_normalizado", "email_canonico", "telefono", "ciudad",
        "codigo_pais", "fecha_alta", "es_activo", "es_email_valido")

    primera = Window.partitionBy("id_cliente").orderBy(F.col("__START_AT._fecha_efectiva"))
    versiones = (historial
                 .withColumn("_orden", F.row_number().over(primera))
                 .select("id_cliente", "segmento",
                         F.when(F.col("_orden") == 1, INICIO_DE_LOS_TIEMPOS)
                          .otherwise(F.col("__START_AT._fecha_efectiva")).alias("vigente_desde"),
                         F.coalesce(F.col("__END_AT._fecha_efectiva"), FIN_DE_LOS_TIEMPOS).alias("vigente_hasta"),
                         F.col("__END_AT").isNull().alias("es_vigente")))

    por_email = Window.partitionBy("email_canonico")
    por_persona = Window.partitionBy("nombre", "apellido", "telefono")
    actual = actual.withColumn(
        "es_posible_duplicado",
        (F.size(F.collect_set("id_cliente").over(por_email)) > 1)
        | (F.col("telefono").isNotNull() & (F.size(F.collect_set("id_cliente").over(por_persona)) > 1)))

    return (versiones.join(actual, "id_cliente", "inner")
            .select(sk("id_cliente", "vigente_desde").alias("sk_cliente"), "id_cliente", "nombre", "apellido",
                    "email_normalizado", "ciudad", "codigo_pais", "segmento", "fecha_alta", "es_activo",
                    "es_email_valido", "es_posible_duplicado", "vigente_desde", "vigente_hasta", "es_vigente"))